# tacvar_visual
## tacvar_visual.draw_heatmap_npb_mpi：对指定测试结果的 region_id / loc_id 绘制多测点热力图

对指定测试结果的 `region_id` 与 `loc_id` 数据绘制多个测点的热力图：横轴为测点序号（按 CSV 行序），纵轴为 MPI 进程号（`rank`），色块为离散分箱颜色（深绿–浅绿–白–浅红–深红）。每个实际存在的 `(region_id, loc_id)` 单独一张图。`data_root` 可以是单个 `data_*` 路径，也可以是路径列表；列表中每个根目录各自出图，按列表顺序拼接。两个及以上根目录时，标题会带上该次运行的目录名。图高固定为 1024 像素；标题与轴标注使用学术印刷风格（Cambria / serif）。

请在 `scripts/` 目录下打开本 notebook，以便 `import tacvar_visual` 可用。CSV 路径形如：

```text
<data_root>/<kernel_name>.<kernel_class>/<short_host>_rRRRR_tTTTT_pPID.csv
```

### 函数原型

```python
tacvar_visual.draw_heatmap_npb_mpi(
    data_root='', kernel_name='', kernel_class='',
    hosts=['all'], region_ids=range(0), loc_ids=range(0),
    fom='elapsed_ns', xrange=-1, ref_section_base=True,
    ref_key='median', ref_base='all',
    bars=[-15, -5, -1, 1, 5, 15],
)
```

`hosts` / `region_ids` / `loc_ids` 及之后的参数为 **keyword-only**。修改模块后请 `importlib.reload` 或重启 kernel。

### 参数说明

| 参数 | 含义 |
|------|------|
| `data_root` / `kernel_name` / `kernel_class` | 定位 `{data_root}/{kernel_name}.{kernel_class}/`。`data_root` 可为字符串或路径列表 |
| `hosts` | `['all']` / `['']` / 空 → 全部主机；否则按 CSV 文件名中的 `short_host` 过滤 |
| `region_ids` | 要绘制的 `region_id`；空可迭代对象（如 `range(0)`）→ 全部 region |
| `loc_ids` | 要绘制的 `loc_id`；空 → 全部 loc；仅绘制数据中实际存在的 (region, loc) 对 |
| `fom` | 数值列名，默认 `elapsed_ns` |
| `xrange` | 要绘制的测点列：`-1` 为全部；`list` 为指定下标；`tuple (a, b)` 为整数切片 `[a, b)`，或两端为 `[0, 1]` 内浮点时取 `[int(n*a), int(n*b))`。整数 `(0, 1)` 只含列 0 |
| `ref_section_base` | `True`：按 `xrange` 窗口计算参考值；`False`：按全部测点计算（图仍只画 `xrange`）。`xrange=-1` 时二者相同 |
| `ref_key` | 参考值统计量：`max` / `min` / `median` / `average`（`mean` 同义）/ `pxx`（百分位，如 `p50`、`p90`） |
| `ref_base` | 参考值样本集合：`all`（全局一个 ref）；`x`（该 seq 的全部 rank）；`y`（该 rank 的 seq，范围受 `ref_section_base` 约束） |
| `bars` | 相对参考的**百分比偏差**切点，须严格递增；分箱数为 `len(bars)+1` |

标题格式：

```text
{fom} heatmap of {Kernel.Class} on {host} (region={region_name}, loc_id={id})
```

多个 `data_root` 时标题为 `({data_dir}; region=..., loc_id=...)`。

色值语义：对每个样本计算 `value / ref`，再落入离散分箱。相对 `ref` 更负 → 更深的绿；包含比值 `1.0`（0% 偏差）的箱 → **始终为白**；更正 → 更深的红。将 `bars` 中的百分比 `p` 转为比值边 `1+p/100`，再补上两端无穷。例如 `bars=[a,b,c]`：

1. `(-inf, a]`
2. `(a, b]`
3. `(b, c]`
4. `(c, +inf)`

默认 `bars=[-15, -5, -1, 1, 5, 15]` 共 7 箱，白箱为 `(-1%, 1%]`。


In [ ]:
import importlib
import matplotlib.pyplot as plt
import tacvar_visual as tvvis
import tacvar_visual.heatmap as _tvh
import tacvar_visual.io as _tvio
importlib.reload(_tvio)
importlib.reload(_tvh)
importlib.reload(tvvis)
plt.close('all')

# Point data_root at a TacVar data_* directory from an NPB-MPI run, e.g.:
# data_root = '../suites/NPB3.4.4/NPB3.4-MPI/data_20260813T082100'
# data_root = ['path/to/data_A', 'path/to/data_B']  # one figure set per root
data_root = [
                '/mnt/keylabmain/nfs/hpckey/03-Project/TacVar_NPC/suites/NPB3.4.4/NPB3.4-MPI/data_20260813T145222',
                '/mnt/keylabmain/nfs/hpckey/03-Project/TacVar_NPC/suites/NPB3.4.4/NPB3.4-MPI/data_20260813T145914'
            ]
kernel_name = 'cg'
kernel_class = 'C'
hosts = ['all']
region_ids = [3]  # empty → all region_ids
loc_ids = []     # empty → all loc_ids
fom = 'elapsed_ns'
# xrange = (0.4, 0.5)
ref_section_base = True
ref_key = 'median'
ref_base = 'all'
bars = [-15, -5, -1, 1, 5, 15]

figs = tvvis.draw_heatmap_npb_mpi(
    data_root,
    kernel_name,
    kernel_class,
    hosts=hosts,
    region_ids=region_ids,
    loc_ids=loc_ids,
    fom=fom,
    xrange=xrange,
    ref_section_base=ref_section_base,
    ref_key=ref_key,
    ref_base=ref_base,
    bars=bars,
)
figs


## tacvar_visual.draw_histogram_npb_mpi：对指定测试结果绘制池化直方图

对指定测试结果的 `region_id` 与 `loc_id` 绘制 `value/ref` 的离散直方图：横轴为与热力图相同的百分比偏差分箱（`bars`，两端 `±inf`），纵轴为样本计数。选定的 MPI `rank` **全部池化**为一组样本，不画分 rank 序列。柱面颜色与热力图一致（深绿–浅绿–白–浅红–深红）；包含 0% 的箱始终为白。每个实际存在的 `(region_id, loc_id)` 单独一张图。`data_root` 同样可以是路径列表，每个根目录各自出图；没有 `overlap` 参数。图高固定为 1024 像素。

请先运行上一格以设定 `data_root` / `fom` / `bars` 等变量。`ranks` 为空可迭代对象时使用全部 rank。

### 函数原型

```python
tacvar_visual.draw_histogram_npb_mpi(
    data_root='', kernel_name='', kernel_class='',
    hosts=['all'], region_ids=range(0), loc_ids=range(0),
    ranks=range(0),
    fom='elapsed_ns', xrange=-1, ref_section_base=True,
    ref_key='median', ref_base='all',
    bars=[-15, -5, -1, 1, 5, 15],
)
```

`hosts` / `region_ids` / `loc_ids` / `ranks` 及之后的参数为 **keyword-only**。`xrange`、`ref_*`、`bars` 语义与热力图相同；`ref_base` 先按格计算 `value/ref`，再把有限值展平。

标题格式：

```text
{fom} histogram of {Kernel.Class} on {host} (region={region_name}, loc_id={id})
```


In [ ]:
import importlib
import matplotlib.pyplot as plt
import tacvar_visual as tvvis
import tacvar_visual.dist as _tvd
import tacvar_visual.heatmap as _tvh
import tacvar_visual.io as _tvio
importlib.reload(_tvio)
importlib.reload(_tvh)
importlib.reload(_tvd)
importlib.reload(tvvis)
plt.close('all')

# Point data_root at a TacVar data_* directory from an NPB-MPI run, e.g.:
# data_root = '../suites/NPB3.4.4/NPB3.4-MPI/data_20260813T082100'
# data_root = ['path/to/data_A', 'path/to/data_B']  # one figure set per root
data_root = '/mnt/keylabmain/nfs/hpckey/03-Project/TacVar_NPC/suites/NPB3.4.4/NPB3.4-MPI/data_20260813T092308'
kernel_name = 'cg'
kernel_class = 'C'
hosts = ['all']
region_ids = [3]  # empty → all region_ids
loc_ids = [2]     # empty → all loc_ids
fom = 'elapsed_ns'
xrange = (0.4, 0.5)
ref_section_base = True
ref_key = 'median'
ref_base = 'all'
bars = [-15, -5, -1, 1, 5, 15]
ranks = []  # empty → all ranks

hist_figs = tvvis.draw_histogram_npb_mpi(
    data_root,
    kernel_name,
    kernel_class,
    hosts=hosts,
    region_ids=region_ids,
    loc_ids=loc_ids,
    ranks=ranks,
    fom=fom,
    xrange=xrange,
    ref_section_base=ref_section_base,
    ref_key=ref_key,
    ref_base=ref_base,
    bars=bars,
)
hist_figs


## tacvar_visual.draw_pdf_npb_mpi：对指定测试结果绘制池化 PDF

对指定测试结果的 `region_id` 与 `loc_id` 绘制 FOM 数值（默认 `elapsed_ns`）的密度直方图（`numpy.histogram(..., density=True)` + stairs）。选定 rank **池化**为一组样本，不画分 rank 曲线。下横轴为 FOM 原值；上横轴为百分比偏差 `bars`（由单个池化 `ref` 线性映射：`value = ref * (1 + p/100)`）。`bars` 同时画成主坐标上的虚线竖线与绿白红色带。纵轴为概率密度。图高 1024 像素。

`data_root` 可为路径或路径列表。`overlap=False`（默认）时每个根目录单独出图，并画上横轴与色带。`overlap=True` 时同一 `(region_id, loc_id)` 的多根目录叠在同一坐标轴上（tab10 颜色、图例为目录名），并**忽略** `bars`（不上横轴、无色带）。

变量沿用上一格；`ranks` 为空表示全部 rank。

### 函数原型

```python
tacvar_visual.draw_pdf_npb_mpi(
    data_root='', kernel_name='', kernel_class='',
    hosts=['all'], region_ids=range(0), loc_ids=range(0),
    ranks=range(0),
    fom='elapsed_ns', xrange=-1, ref_section_base=True,
    ref_key='median', ref_base='all',
    bars=[-15, -5, -1, 1, 5, 15],
    overlap=False,
)
```

标题格式：

```text
{fom} pdf of {Kernel.Class} on {host} (region={region_name}, loc_id={id})
```


In [ ]:
import importlib
import matplotlib.pyplot as plt
import tacvar_visual as tvvis
import tacvar_visual.dist as _tvd
import tacvar_visual.heatmap as _tvh
import tacvar_visual.io as _tvio
importlib.reload(_tvio)
importlib.reload(_tvh)
importlib.reload(_tvd)
importlib.reload(tvvis)
plt.close('all')

ranks = []  # empty → all ranks
# data_root may be a string or a list of data_* paths
data_root = [
                '/mnt/keylabmain/nfs/hpckey/03-Project/TacVar_NPC/suites/NPB3.4.4/NPB3.4-MPI/data_20260813T145222',
                '/mnt/keylabmain/nfs/hpckey/03-Project/TacVar_NPC/suites/NPB3.4.4/NPB3.4-MPI/data_20260813T145914'
            ]
kernel_name = 'cg'
kernel_class = 'C'
hosts = ['all']
region_ids = [3]  # empty → all region_ids
loc_ids = [1]     # empty → all loc_ids
fom = 'elapsed_ns'
xrange = -1
ref_section_base = True
ref_key = 'median'
ref_base = 'all'
bars = [-15, -5, -1, 1, 5, 15]
ranks = []  # empty → all ranks
overlap = True  # True: overlay listed roots; bars ignored

pdf_figs = tvvis.draw_pdf_npb_mpi(
    data_root,
    kernel_name,
    kernel_class,
    hosts=hosts,
    region_ids=region_ids,
    loc_ids=loc_ids,
    ranks=ranks,
    fom=fom,
    xrange=xrange,
    ref_section_base=ref_section_base,
    ref_key=ref_key,
    ref_base=ref_base,
    bars=bars,
    overlap=overlap,
)
pdf_figs


## tacvar_visual.draw_cdf_npb_mpi：对指定测试结果绘制池化 CDF

对指定测试结果的 `region_id` 与 `loc_id` 绘制 FOM 数值的经验 CDF（排序后 `arange(1, n+1)/n`，阶梯曲线）。选定 rank **池化**为一组样本，不画分 rank 曲线。坐标轴与 PDF 相同：下横轴为 FOM，上横轴为百分比 `bars`；`overlap=False` 时主坐标上仍有虚线竖线与绿白红色带。纵轴为 `[0, 1]` 累积概率；0.5 处有水平点线。图高 1024 像素。

`overlap=True` 时多根目录以不同颜色的阶梯曲线叠在同一坐标轴上，并忽略 `bars`；0.5 水平点线保留。

变量沿用上一格；`ranks` 为空表示全部 rank。

### 函数原型

```python
tacvar_visual.draw_cdf_npb_mpi(
    data_root='', kernel_name='', kernel_class='',
    hosts=['all'], region_ids=range(0), loc_ids=range(0),
    ranks=range(0),
    fom='elapsed_ns', xrange=-1, ref_section_base=True,
    ref_key='median', ref_base='all',
    bars=[-15, -5, -1, 1, 5, 15],
    overlap=False,
)
```

标题格式：

```text
{fom} cdf of {Kernel.Class} on {host} (region={region_name}, loc_id={id})
```


In [ ]:
import importlib
import matplotlib.pyplot as plt
import tacvar_visual as tvvis
import tacvar_visual.dist as _tvd
import tacvar_visual.heatmap as _tvh
import tacvar_visual.io as _tvio
importlib.reload(_tvio)
importlib.reload(_tvh)
importlib.reload(_tvd)
importlib.reload(tvvis)
plt.close('all')

ranks = []  # empty → all ranks
# List of roots + overlap=True overlays CDF curves on one axes (bars ignored)
data_root = [
                '/mnt/keylabmain/nfs/hpckey/03-Project/TacVar_NPC/suites/NPB3.4.4/NPB3.4-MPI/data_20260813T145222',
                '/mnt/keylabmain/nfs/hpckey/03-Project/TacVar_NPC/suites/NPB3.4.4/NPB3.4-MPI/data_20260813T145914'
            ]
kernel_name = 'cg'
kernel_class = 'C'
hosts = ['all']
region_ids = [3]  # empty → all region_ids
loc_ids = [1]     # empty → all loc_ids
fom = 'elapsed_ns'
xrange = -1
ref_section_base = True
ref_key = 'median'
ref_base = 'all'
bars = [-15, -5, -1, 1, 5, 15]
ranks = []  # empty → all ranks
overlap = True

cdf_figs = tvvis.draw_cdf_npb_mpi(
    data_root,
    kernel_name,
    kernel_class,
    hosts=hosts,
    region_ids=region_ids,
    loc_ids=loc_ids,
    ranks=ranks,
    fom=fom,
    xrange=xrange,
    ref_section_base=ref_section_base,
    ref_key=ref_key,
    ref_base=ref_base,
    bars=bars,
    overlap=overlap,
)
cdf_figs
